# Data Snapshot Metadata Extraction — Schema v1.3

Extract Pydantic-validated metadata from snapshot images and append one record per attempt to JSONL. Run the selection cells before the live-call cell.

Successful paths already present in the selected results file are excluded before sampling. Set `EXACT_FILE_PATHS` to diagnose chosen snapshots; when nonempty, it takes precedence over `SOURCES`, `TYPES`, and `n_samples`. Use new calibration filenames for each test, then switch to `results.jsonl` and `errors.jsonl` for the full run.

In [1]:
%load_ext autotime

import json
import random
import time
from pathlib import Path

from tqdm.auto import tqdm

from data_snapshot.constants import ROOT
from data_snapshot.metadata_extraction import extract_metadata

## Inputs

In [2]:
SOURCES = ["unhcr", "prwp", "refugee"]
TYPES = ["figure", "table"]
n_samples = 1  # Per source and type; use None for every eligible snapshot.

# Absolute paths or paths relative to ROOT. Nonempty means exact-path mode.
EXACT_FILE_PATHS = []

SLEEP_SECONDS = 0.2
NOTEBOOK_DIR = ROOT / "notebooks/metadata_extraction"
SNAPSHOTS_DIR = NOTEBOOK_DIR / "data/snapshots"
CONFIG_PATH = ROOT / "src/data_snapshot/metadata_extraction/config/default.json"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
RESULTS_PATH = OUTPUT_DIR / "calibration0_results.jsonl"
ERRORS_PATH = OUTPUT_DIR / "calibration0_errors.jsonl"

## Selection helpers

In [3]:
def snapshot_key(path: Path) -> str:
    """Return a stable source/type/filename key for a snapshot path."""
    try:
        relative = path.resolve().relative_to(SNAPSHOTS_DIR.resolve())
    except ValueError as exc:
        raise ValueError(f"Snapshot is outside {SNAPSHOTS_DIR}: {path}") from exc
    if len(relative.parts) != 3 or relative.parts[1] not in {"figure", "table"}:
        raise ValueError(f"Unexpected snapshot layout: {relative}")
    return relative.as_posix()


def resolve_exact_path(value: str | Path) -> Path:
    """Resolve and validate one absolute or repository-relative PNG path."""
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = ROOT / path
    path = path.resolve()
    if not path.is_file() or path.suffix.lower() != ".png":
        raise ValueError(f"Exact snapshot path is not a PNG file: {path}")
    snapshot_key(path)
    return path


def load_completed_paths(path: Path) -> set[str]:
    """Read successful snapshot keys from an existing results JSONL."""
    if not path.exists():
        return set()
    completed = set()
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on {path}:{line_number}") from exc
            key = record.get("snapshot_path")
            if not isinstance(key, str) or not key:
                raise ValueError(f"Missing snapshot_path on {path}:{line_number}")
            completed.add(key)
    return completed


def append_jsonl(path: Path, record: dict[str, object]) -> None:
    """Append one JSON-compatible record, creating its directory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

## Build the skip list before selection

In [4]:
completed_snapshot_paths = load_completed_paths(RESULTS_PATH)
print(f"Completed snapshots in {RESULTS_PATH.name}: {len(completed_snapshot_paths)}")

Completed snapshots in calibration0_results.jsonl: 0


## Select snapshots

In [5]:
selected_snapshots = []

if EXACT_FILE_PATHS:
    seen = set()
    for value in EXACT_FILE_PATHS:
        path = resolve_exact_path(value)
        key = snapshot_key(path)
        if key in completed_snapshot_paths:
            print(f"Already completed: {key}")
        elif key not in seen:
            selected_snapshots.append(path)
            seen.add(key)
    selection_mode = "exact paths"
else:
    if n_samples is not None and (not isinstance(n_samples, int) or n_samples < 0):
        raise ValueError("n_samples must be a non-negative integer or None.")
    for source in SOURCES:
        for artifact_type in TYPES:
            directory = SNAPSHOTS_DIR / source / artifact_type
            eligible = [
                path
                for path in sorted(directory.glob("*.png"))
                if snapshot_key(path) not in completed_snapshot_paths
            ]
            count = (
                len(eligible) if n_samples is None else min(n_samples, len(eligible))
            )
            sampled = eligible if n_samples is None else random.sample(eligible, count)
            selected_snapshots.extend(sampled)
            print(
                f"{source}/{artifact_type}: {len(eligible)} eligible, "
                f"{len(sampled)} selected"
            )
    selected_snapshots.sort(key=snapshot_key)
    selection_mode = "stratified sample"

print(f"Selection mode: {selection_mode}")
print(f"Total selected: {len(selected_snapshots)}")
for path in selected_snapshots:
    print(snapshot_key(path))

unhcr/figure: 17 eligible, 1 selected
unhcr/table: 17 eligible, 1 selected
prwp/figure: 17 eligible, 1 selected
prwp/table: 17 eligible, 1 selected
refugee/figure: 17 eligible, 1 selected
refugee/table: 17 eligible, 1 selected
Selection mode: stratified sample
Total selected: 6
prwp/figure/document_11174028_figure_002.png
prwp/table/document_11174028_table_007.png
refugee/figure/027_Jordan-Emergency-Food-Security-Project_figure_000.png
refugee/table/001_BOSIB-3f2311b3-9a20-44d3-b637-b3b2b3d21695_table_008.png
unhcr/figure/impact_lby_so_refugees_and_migrants_access_to_food_wash_shelter_november_2018_figure_007.png
unhcr/table/rpublique_dmocratique_du_congo_-_points_saillants_de_protection_-_aot_2024_table_000.png


## Live API calls

Running the next cell incurs API usage. Successful records go to `RESULTS_PATH`; failures remain retryable and go to `ERRORS_PATH`.

In [6]:
succeeded = failed = skipped = 0

for snapshot_path in tqdm(selected_snapshots, unit="snapshot"):
    key = snapshot_key(snapshot_path)
    if key in completed_snapshot_paths:
        skipped += 1
        continue

    source, artifact_type, _ = key.split("/", maxsplit=2)
    result = extract_metadata(snapshot_path, config_path=CONFIG_PATH)
    record = {
        "snapshot_path": key,
        "snapshot_file_name": snapshot_path.name,
        "source": source,
        "artifact_type": artifact_type,
        "schema_version": "1.3",
        "model": result.model,
        "response_id": result.response_id,
        "api_status": result.api_status,
        "elapsed_seconds": result.elapsed_seconds,
        "usage": result.usage,
    }
    if result.metadata is not None:
        append_jsonl(
            RESULTS_PATH,
            {
                **record,
                "metadata": result.metadata.model_dump(mode="json", exclude_none=True),
            },
        )
        completed_snapshot_paths.add(key)
        succeeded += 1
    else:
        append_jsonl(
            ERRORS_PATH,
            {
                **record,
                "raw_output": result.raw_output,
                "error_type": result.error_type,
                "error": result.error,
            },
        )
        failed += 1
        print(f"{key}: {result.error}")

    if result.elapsed_seconds is not None and SLEEP_SECONDS:
        time.sleep(SLEEP_SECONDS)

  0%|          | 0/6 [00:00<?, ?snapshot/s]

In [7]:
print(f"Succeeded: {succeeded}")
print(f"Failed: {failed}")
print(f"Skipped after selection: {skipped}")
print(f"Results: {RESULTS_PATH}")
print(f"Errors: {ERRORS_PATH}")

Succeeded: 6
Failed: 0
Skipped after selection: 0
Results: /home/ajd/data-snapshot-annotation/notebooks/metadata_extraction/outputs/calibration0_results.jsonl
Errors: /home/ajd/data-snapshot-annotation/notebooks/metadata_extraction/outputs/calibration0_errors.jsonl
